# Pregunta 3 — Entrenamiento YOLO en CCT-20 (Colab / Kaggle)

Flujo: entorno → código → datos → exploración → conversión → E1/E2 (vacías) → E3 (modelo principal) → evaluación cis/trans.

**Antes de correr**
- **Colab:** Entorno de ejecución → Cambiar tipo → GPU (T4). Los resultados se guardan en Google Drive (`MyDrive/pregunta3_runs`) para no perderlos si se corta la sesión.
- **Kaggle:** Settings → Accelerator: GPU T4 / P100, y **Internet: On**. Los resultados quedan en `/kaggle/working/runs`.
- Si la sesión se corta a mitad de entrenamiento, vuelve a correr todo: la descarga se salta lo que ya existe y el entrenamiento se reanuda desde `last.pt`.

In [ ]:
# 1. Entorno y rutas
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
EN_KAGGLE = Path("/kaggle").exists()

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CODE = Path("/content/pregunta3")
    DATA = Path("/content/data")                      # disco local: rapido
    RUNS = Path("/content/drive/MyDrive/pregunta3_runs")
elif EN_KAGGLE:
    import shutil
    CODE = Path("/kaggle/working/pregunta3")
    DATA = Path("/tmp/cct")                           # /kaggle/working solo tiene ~20 GB
    RUNS = Path("/kaggle/working/runs")
    # continuar lo entrenado en Colab: la carpeta pregunta3_runs de Drive subida como Dataset
    previos = sorted({p.parents[2] for p in Path("/kaggle/input").rglob("weights/last.pt")})
    if previos and not RUNS.exists():
        shutil.copytree(previos[0], RUNS)
        print("resultados previos copiados desde", previos[0])
    # los checkpoints guardan rutas de Colab: se replican con enlaces para poder reanudar
    Path("/content/drive/MyDrive").mkdir(parents=True, exist_ok=True)
    for enlace, real in [("/content/data", DATA), ("/content/drive/MyDrive/pregunta3_runs", RUNS)]:
        if not os.path.lexists(enlace):
            os.symlink(real, enlace)
else:
    CODE = Path.cwd()
    DATA = CODE / "data"
    RUNS = CODE / "runs"

for p in (DATA, RUNS):
    p.mkdir(parents=True, exist_ok=True)
WORKERS = 2 if EN_COLAB else 4
print("colab:", EN_COLAB, "| kaggle:", EN_KAGGLE)
print("codigo:", CODE, "\ndatos:", DATA, "\nresultados:", RUNS)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Código del proyecto
Opción A: pon la URL de tu repositorio de GitHub en `REPO_URL`.
Opción B: deja `REPO_URL` vacío y sube `pregunta3_codigo.zip` (en Colab: panel de archivos → `/content`; en Kaggle: como Dataset, queda en `/kaggle/input/...`).

In [ ]:
REPO_URL = "https://github.com/roshan0297/rnc-pregunta3-parcial.git"
SUBCARPETA = ""   # carpeta dentro del repo donde esta src/ ("" = raiz)

if not (CODE / "src").exists():
    if REPO_URL:
        tmp = Path("/tmp/repo")
        if not tmp.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(tmp)], check=True)
        src_repo = tmp / SUBCARPETA if (tmp / SUBCARPETA).exists() else tmp
        subprocess.run(["cp", "-r", str(src_repo), str(CODE)], check=True)
    else:
        zips = list(Path("/content").glob("pregunta3_codigo.zip")) + list(Path("/kaggle/input").rglob("pregunta3_codigo.zip"))
        assert zips, "Sube pregunta3_codigo.zip o pon REPO_URL"
        subprocess.run(["unzip", "-oq", str(zips[0]), "-d", str(CODE)], check=True)

os.chdir(CODE)
sys.path.insert(0, str(CODE / "src"))
!ls {CODE} {CODE}/src

In [ ]:
# 3. Dependencias
!pip install -q -r requirements.txt
import ultralytics
ultralytics.checks()

## 4. Descarga de datos
Imágenes CCT-20 (~6 GB) + 3000 vacías de ubicaciones que no están en CCT-20 (el split `train` no trae vacías). En Colab tarda ~10–20 min.

In [ ]:
RAW = DATA / "raw"
!python src/descargar_datos.py --raw {RAW} --vacias-extra 3000 --borrar-tar
!du -sh {RAW}/*

In [ ]:
# 5. Exploracion con imagenes (porcentaje nocturno y mosaico de ejemplos)
!python src/explorar.py --anot {RAW}/eccv_18_annotation_files --imgs {RAW}/imagenes --out {RUNS}/exploracion

## 6. Conversión a YOLO
Dos versiones del dataset para el experimento de vacías: sin vacías y con 20 % de vacías extra. Val (`cis_val` + `trans_val`) y test son iguales en ambas. Las imágenes se enlazan (hardlink), no se copian.

In [ ]:
YOLO_SIN = DATA / "yolo_sin_vacias"
YOLO_V20 = DATA / "yolo_vacias20"
COMUNES = f"--anot {RAW}/eccv_18_annotation_files --imgs {RAW}/imagenes --vacias-extra {RAW}/vacias_extra"

!python src/convertir_yolo.py {COMUNES} --salida {YOLO_SIN} --frac-vacias 0
!python src/convertir_yolo.py {COMUNES} --salida {YOLO_V20} --frac-vacias 0.2

# guardar los metadatos de la conversion junto a los resultados
!mkdir -p {RUNS}/conversion && cp {YOLO_V20}/metadatos.csv {YOLO_V20}/config_conversion.json {RUNS}/conversion/

## 7. Entrenamiento
Hiperparámetros comunes. Cada decisión debe quedar justificada en el informe:
- `flipud=0`: los animales no aparecen de cabeza.
- `hsv_s` bajo (0.4 en vez de 0.7): buena parte de las fotos son infrarrojas (grises) y saturar color no aporta.
- `patience=10`: corta si el mAP de validación no mejora en 10 épocas.

In [ ]:
import pandas as pd
from ultralytics import YOLO

COMUN = dict(
    imgsz=640, epochs=30, batch=32, workers=WORKERS, device=0, seed=42,
    patience=10, project=str(RUNS), exist_ok=True,
    fliplr=0.5, flipud=0.0, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,
    mosaic=1.0, close_mosaic=5,
)

def entrenar(nombre, yaml, modelo="yolov8n.pt", **cambios):
    # Entrena o reanuda. Devuelve la ruta de best.pt
    pesos = RUNS / nombre / "weights"
    if (pesos / "last.pt").exists():
        try:
            YOLO(pesos / "last.pt").train(resume=True)
        except AssertionError as e:   # ya habia terminado: no hay nada que reanudar
            print("ya entrenado:", e)
    else:
        YOLO(modelo).train(data=str(yaml), name=nombre, **{**COMUN, **cambios})
    return pesos / "best.pt"


def tasa_falsos_disparos(pesos, dir_yolo, splits=("cis_val", "trans_val"), conf=0.5, imgsz=640):
    # % de imagenes vacias donde el modelo detecta algo con confianza >= conf
    modelo = YOLO(pesos)
    filas = []
    for s in splits:
        vacias = [dir_yolo / "images" / s / (t.stem + ".jpg")
                  for t in (dir_yolo / "labels" / s).glob("*.txt") if t.stat().st_size == 0]
        disparos = 0
        # en lotes de 32: una lista grande de rutas se procesa como un solo lote y llena la GPU
        for i in range(0, len(vacias), 32):
            res = modelo.predict(vacias[i:i + 32], conf=conf, imgsz=imgsz, verbose=False)
            disparos += sum(len(r.boxes) > 0 for r in res)
        filas.append({"split": s, "n_vacias": len(vacias), "con_disparo": disparos,
                      "tasa": disparos / max(len(vacias), 1)})
    return pd.DataFrame(filas)

### E1 y E2: ¿agregar vacías ayuda?
yolov8n (rápido, ~1 h cada uno en T4). Misma configuración; lo único que cambia son las vacías en train.

In [ ]:
best_e1 = entrenar("E1_n640_sin_vacias", YOLO_SIN / "cct.yaml")

In [ ]:
best_e2 = entrenar("E2_n640_vacias20", YOLO_V20 / "cct.yaml")

In [ ]:
# Comparacion E1 vs E2 sobre validacion: mAP y falsos disparos en vacias
filas = []
for nombre, best, d in [("E1", best_e1, YOLO_SIN), ("E2", best_e2, YOLO_V20)]:
    m = YOLO(best).val(data=str(d / "cct.yaml"), split="val", imgsz=640, verbose=False, plots=False)
    fd = tasa_falsos_disparos(best, d)
    filas.append({"exp": nombre, "mAP50": m.box.map50, "mAP50-95": m.box.map,
                  **{f"falsos_{r.split}": r.tasa for r in fd.itertuples()}})
comp = pd.DataFrame(filas)
comp.to_csv(RUNS / "comparacion_E1_E2.csv", index=False)
comp

### E3: modelo principal
Elige la configuración según E1/E2 (y el histograma de tamaños de la exploración) y anota el porqué en la bitácora.
`yolov8s` a 640 con 50 épocas tarda ~4–6 h en T4: lánzalo y deja la pestaña abierta (Colab) o usa "Save & Run All" (Kaggle).

In [ ]:
DATASET_E3 = YOLO_V20      # o YOLO_SIN, segun E1 vs E2
MODELO_E3 = "yolov8s.pt"
IMGSZ_E3 = 640             # 960 si las cajas pequenas pesan mucho y la GPU lo permite (batch 8)

best_e3 = entrenar("E3_principal", DATASET_E3 / "cct.yaml", MODELO_E3,
                   imgsz=IMGSZ_E3, epochs=50, batch=16 if IMGSZ_E3 == 640 else 8)

## 8. Evaluación final: cis_test vs trans_test
mAP@0.5 y mAP@0.5:0.95 global y por especie, más la tasa de falsos disparos en las vacías de test. Esto alimenta los entregables 2 y 3.

In [ ]:
def evaluar(best, dir_yolo, imgsz, nombre_exp):
    tablas = []
    for yaml, test in [("cct.yaml", "cis_test"), ("cct_trans.yaml", "trans_test")]:
        m = YOLO(best).val(data=str(dir_yolo / yaml), split="test", imgsz=imgsz,
                           project=str(RUNS / nombre_exp), name=f"eval_{test}", exist_ok=True)
        t = pd.DataFrame({
            "especie": [m.names[i] for i in m.box.ap_class_index],
            "AP50": m.box.ap50, "AP50-95": m.box.ap,
        })
        t.loc[len(t)] = ["TODAS (mAP)", m.box.map50, m.box.map]
        t.insert(0, "test", test)
        tablas.append(t)
    res = pd.concat(tablas)
    res.to_csv(RUNS / nombre_exp / "eval_por_especie.csv", index=False)
    fd = tasa_falsos_disparos(best, dir_yolo, splits=("cis_test", "trans_test"), imgsz=imgsz)
    fd.to_csv(RUNS / nombre_exp / "falsos_disparos_test.csv", index=False)
    return res, fd

res, fd = evaluar(best_e3, DATASET_E3, IMGSZ_E3, "E3_principal")
display(res.pivot(index="especie", columns="test", values="AP50").round(3))
display(fd)

## 9. Análisis (entregables 2–5)
Se corre sobre el modelo final. Si E3 no alcanzó a terminar, usa E2 y anótalo como limitación de cómputo.
Antes, sube a GitHub `results/oclusion/etiquetas_oclusion.csv` (lo generas en tu PC con `src/etiquetar_oclusion.py`): la primera celda trae la última versión del repo.

In [ ]:
# traer scripts nuevos y las etiquetas de oclusion desde GitHub
!git -C /tmp/repo pull -q && cp -r /tmp/repo/src /tmp/repo/results {CODE}/ && ls {CODE}/src {CODE}/results

EXP_FINAL = "E3_principal" if (RUNS / "E3_principal/weights/best.pt").exists() else "E2_n640_vacias20"
DATASET_FINAL = YOLO_V20     # el dataset con el que se entreno EXP_FINAL
IMGSZ_FINAL = 640
BEST = RUNS / EXP_FINAL / "weights" / "best.pt"
OUT = RUNS / EXP_FINAL
ANOT = RAW / "eccv_18_annotation_files"
IMGS = RAW / "imagenes"
print("modelo final:", BEST)

**Umbral de confianza.** Mira `F1_curve.png` en la carpeta de entrenamiento de `EXP_FINAL` (se calcula sobre validación) y pon en `CONF` la confianza donde la curva "all classes" es máxima. Elegirlo en validación y no en test evita hacer trampa.

In [ ]:
from IPython.display import Image as Img
display(Img(str(OUT / "F1_curve.png"), width=600))
CONF = 0.5   # <- ajusta segun la curva

In [ ]:
# predicciones sobre todo cis_test y trans_test (una sola vez, ~10-15 min); las usan los analisis siguientes
PRED = OUT / "predicciones.csv"
if not PRED.exists():
    !python src/predecir.py --pesos {BEST} --anot {ANOT} --imgs {IMGS} --imgsz {IMGSZ_FINAL} --salida {PRED}

In [ ]:
# Entregable 2: AP por especie vs numero de cajas de entrenamiento
!python src/analisis_desbalance.py --eval {OUT}/eval_por_especie.csv --yolo {DATASET_FINAL} --out {OUT}/desbalance
display(Img(str(OUT / "desbalance/ap_vs_cajas_train.png"), width=600))

In [ ]:
# Entregable 4: recall por tamano y por oclusion
!python src/tamano_oclusion.py --pred {PRED} --anot {ANOT} --conf {CONF} --oclusion results/oclusion/etiquetas_oclusion.csv --out {OUT}/tamano_oclusion
display(Img(str(OUT / "tamano_oclusion/recall_tamano_oclusion.png"), width=800))

In [ ]:
# Entregable 3: EigenCAM sobre el backbone (capa 9 = SPPF en YOLOv8), cis vs trans
!python src/eigencam.py --pesos {BEST} --anot {ANOT} --imgs {IMGS} --imgsz {IMGSZ_FINAL} --conf {CONF} --out {OUT}/eigencam
display(Img(str(OUT / "eigencam/eigencam_ejemplos.png"), width=1000))

In [ ]:
# Entregable 3: prueba de borrado (se tapa el animal y se vuelve a detectar)
!python src/prueba_borrado.py --pesos {BEST} --anot {ANOT} --imgs {IMGS} --imgsz {IMGSZ_FINAL} --conf {CONF} --out {OUT}/borrado
display(Img(str(OUT / "borrado/borrado_ejemplos.png"), width=1000))

### Entregable 5: monitoreo ecológico
La primera celda lista las cámaras con cuántas fotos tiene cada una. Elige una de `trans_test` (nunca vista en entrenamiento) y justifica la elección (p. ej. la de más fotos, o una con varias especies).

In [ ]:
!python src/monitoreo.py --pred {PRED} --anot {ANOT} --out {OUT}/monitoreo

In [ ]:
CAMARA = "46"   # <- elige segun la tabla anterior
!python src/monitoreo.py --pred {PRED} --anot {ANOT} --conf {CONF} --ubicacion {CAMARA} --out {OUT}/monitoreo
display(pd.read_csv(OUT / f"monitoreo/camara_{CAMARA}/reporte_especies.csv"))
display(Img(str(OUT / f"monitoreo/camara_{CAMARA}/actividad_hora.png"), width=1000))